In [ ]:
import os
import cv2
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from collections import Counter
from scipy.stats import mode

from skimage.feature import hog, local_binary_pattern

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("Libraries loaded successfully!")

In [ ]:
import kagglehub

path = kagglehub.dataset_download("moltean/fruits")

print("Dataset path:")
print(path)

In [ ]:
dataset_100 = os.path.join(
    path,
    "fruits-360_100x100",
    "fruits-360"
)

training_path = os.path.join(
    dataset_100,
    "Training"
)

test_path = os.path.join(
    dataset_100,
    "Test"
)

print("Training path:", training_path)
print("Test path:", test_path)

print("\nTraining exists:", os.path.exists(training_path))
print("Test exists:", os.path.exists(test_path))

In [ ]:
selected_classes = [
    "Apple 10",
    "Banana 1",
    "Cherry 1",
    "Grape 1",
    "Kiwi 1",
    "Mango 1",
    "Orange 1",
    "Peach 1",
    "Pear 1",
    "Strawberry 1"
]

print("Selected classes:")
for i, cls in enumerate(selected_classes):
    print(i, "->", cls)

In [ ]:
IMG_SIZE = 100

X_train = []
y_train = []

X_test = []
y_test = []

print("Loading training images...")

for label, class_name in enumerate(selected_classes):

    class_path = os.path.join(training_path, class_name)

    for filename in os.listdir(class_path):

        image_path = os.path.join(class_path, filename)

        image = cv2.imread(image_path)

        if image is not None:

            image = cv2.cvtColor(
                image,
                cv2.COLOR_BGR2RGB
            )

            image = cv2.resize(
                image,
                (IMG_SIZE, IMG_SIZE)
            )

            X_train.append(image)
            y_train.append(label)


print("Loading test images...")

for label, class_name in enumerate(selected_classes):

    class_path = os.path.join(test_path, class_name)

    for filename in os.listdir(class_path):

        image_path = os.path.join(class_path, filename)

        image = cv2.imread(image_path)

        if image is not None:

            image = cv2.cvtColor(
                image,
                cv2.COLOR_BGR2RGB
            )

            image = cv2.resize(
                image,
                (IMG_SIZE, IMG_SIZE)
            )

            X_test.append(image)
            y_test.append(label)


X_train = np.array(X_train)
y_train = np.array(y_train)

X_test = np.array(X_test)
y_test = np.array(y_test)

print("\nDataset loaded successfully!")

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

In [ ]:
def augment_image(image):
    """
    Create one realistic augmented version
    of a fruit image.
    """

    img = image.copy()

    # -----------------------------
    # 1. Random horizontal flip
    # -----------------------------
    if np.random.rand() < 0.5:
        img = cv2.flip(img, 1)

    # -----------------------------
    # 2. Small rotation
    # -----------------------------
    angle = np.random.uniform(-15, 15)

    center = (
        IMG_SIZE // 2,
        IMG_SIZE // 2
    )

    matrix = cv2.getRotationMatrix2D(
        center,
        angle,
        1.0
    )

    img = cv2.warpAffine(
        img,
        matrix,
        (IMG_SIZE, IMG_SIZE),
        borderMode=cv2.BORDER_REFLECT_101
    )

    # -----------------------------
    # 3. Brightness / contrast
    # -----------------------------
    alpha = np.random.uniform(0.85, 1.15)
    beta = np.random.uniform(-20, 20)

    img = cv2.convertScaleAbs(
        img,
        alpha=alpha,
        beta=beta
    )

    # -----------------------------
    # 4. Slight blur sometimes
    # -----------------------------
    if np.random.rand() < 0.20:
        img = cv2.GaussianBlur(
            img,
            (3, 3),
            0
        )

    return img

In [ ]:
sample_index = 0

original = X_train[sample_index]

augmented_images = [
    augment_image(original)
    for _ in range(5)
]

plt.figure(figsize=(15, 3))

plt.subplot(1, 6, 1)
plt.imshow(original)
plt.title("Original")
plt.axis("off")

for i, img in enumerate(augmented_images):

    plt.subplot(1, 6, i + 2)
    plt.imshow(img)
    plt.title(f"Augmented {i+1}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)

X_augmented = []
y_augmented = []

print("Creating augmented training images...")

for image, label in tqdm(
    zip(X_train, y_train),
    total=len(X_train)
):

    augmented = augment_image(image)

    X_augmented.append(augmented)
    y_augmented.append(label)


X_augmented = np.array(X_augmented)
y_augmented = np.array(y_augmented)

print("\nAugmentation completed!")

print("Original training images:",
      len(X_train))

print("Augmented images:",
      len(X_augmented))

In [ ]:
X_train_final = np.concatenate(
    [X_train, X_augmented],
    axis=0
)

y_train_final = np.concatenate(
    [y_train, y_augmented],
    axis=0
)

print("Final training dataset:")
print("X_train_final:", X_train_final.shape)
print("y_train_final:", y_train_final.shape)

print("\nTest dataset remains:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

In [ ]:
# ============================================================
# CELL 12 - FEATURE EXTRACTION
# ============================================================

def extract_features(image):

    # --------------------------------------------------------
    # COLOR FEATURES
    # --------------------------------------------------------

    # RGB statistics
    rgb_mean = np.mean(image, axis=(0, 1))
    rgb_std = np.std(image, axis=(0, 1))

    # HSV statistics
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)

    hsv_mean = np.mean(hsv, axis=(0, 1))
    hsv_std = np.std(hsv, axis=(0, 1))

    # --------------------------------------------------------
    # COLOR HISTOGRAMS
    # --------------------------------------------------------

    color_hist = []

    for channel in range(3):

        hist = cv2.calcHist(
            [image],
            [channel],
            None,
            [16],
            [0, 256]
        )

        hist = cv2.normalize(
            hist,
            hist
        ).flatten()

        color_hist.extend(hist)

    # --------------------------------------------------------
    # HOG FEATURES
    # --------------------------------------------------------

    gray = cv2.cvtColor(
        image,
        cv2.COLOR_RGB2GRAY
    )

    hog_features = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys'
    )

    # --------------------------------------------------------
    # LBP FEATURES
    # --------------------------------------------------------

    radius = 1
    n_points = 8 * radius

    lbp = local_binary_pattern(
        gray,
        n_points,
        radius,
        method='uniform'
    )

    n_bins = n_points + 2

    lbp_hist, _ = np.histogram(
        lbp.ravel(),
        bins=np.arange(0, n_bins + 1),
        range=(0, n_bins)
    )

    lbp_hist = lbp_hist.astype("float")
    lbp_hist /= (lbp_hist.sum() + 1e-7)

    # --------------------------------------------------------
    # COMBINE ALL FEATURES
    # --------------------------------------------------------

    features = np.concatenate([
        rgb_mean,
        rgb_std,
        hsv_mean,
        hsv_std,
        color_hist,
        hog_features,
        lbp_hist
    ])

    return features.astype(np.float32)

In [ ]:
# ============================================================
# CELL 13 - EXTRACT FEATURES
# ============================================================

print("Extracting features from augmented training dataset...")

X_train_features = np.array([
    extract_features(img)
    for img in tqdm(X_train_final)
])

print("\nExtracting features from test dataset...")

X_test_features = np.array([
    extract_features(img)
    for img in tqdm(X_test)
])

print("\nFeature extraction completed!")

print("X_train_features shape:",
      X_train_features.shape)

print("X_test_features shape:",
      X_test_features.shape)

In [ ]:
# ============================================================
# CELL 14 - FEATURE SCALING
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train_features
)

X_test_scaled = scaler.transform(
    X_test_features
)

print("Feature scaling completed!")

print("X_train_scaled:",
      X_train_scaled.shape)

print("X_test_scaled:",
      X_test_scaled.shape)

In [ ]:
# ============================================================
# CELL 15 - TRAIN SVM
# ============================================================

print("Training SVM...")

start_time = time.time()

svm_model = SVC(
    kernel='rbf',
    C=10,
    gamma='scale',
    probability=True,
    random_state=42
)

svm_model.fit(
    X_train_scaled,
    y_train_final
)

svm_time = time.time() - start_time

svm_pred = svm_model.predict(
    X_test_scaled
)

svm_accuracy = accuracy_score(
    y_test,
    svm_pred
)

print(f"SVM Accuracy: {svm_accuracy * 100:.2f}%")
print(f"Training Time: {svm_time:.2f} seconds")

In [ ]:
# ============================================================
# CELL 16 - TRAIN KNN
# ============================================================

print("Training KNN...")

start_time = time.time()

knn_model = KNeighborsClassifier(
    n_neighbors=5,
    weights='distance',
    n_jobs=-1
)

knn_model.fit(
    X_train_scaled,
    y_train_final
)

knn_time = time.time() - start_time

knn_pred = knn_model.predict(
    X_test_scaled
)

knn_accuracy = accuracy_score(
    y_test,
    knn_pred
)

print(f"KNN Accuracy: {knn_accuracy * 100:.2f}%")
print(f"Training Time: {knn_time:.2f} seconds")

In [ ]:
# ============================================================
# CELL 17 - TRAIN DECISION TREE
# ============================================================

print("Training Decision Tree...")

start_time = time.time()

decision_tree_model = DecisionTreeClassifier(
    max_depth=30,
    min_samples_split=2,
    random_state=42
)

decision_tree_model.fit(
    X_train_scaled,
    y_train_final
)

dt_time = time.time() - start_time

dt_pred = decision_tree_model.predict(
    X_test_scaled
)

dt_accuracy = accuracy_score(
    y_test,
    dt_pred
)

print(f"Decision Tree Accuracy: {dt_accuracy * 100:.2f}%")
print(f"Training Time: {dt_time:.2f} seconds")

In [ ]:
# ============================================================
# CELL 18 - TRAIN RANDOM FOREST
# ============================================================

print("Training Random Forest...")

start_time = time.time()

random_forest_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=30,
    n_jobs=-1,
    random_state=42
)

random_forest_model.fit(
    X_train_scaled,
    y_train_final
)

rf_time = time.time() - start_time

rf_pred = random_forest_model.predict(
    X_test_scaled
)

rf_accuracy = accuracy_score(
    y_test,
    rf_pred
)

print(f"Random Forest Accuracy: {rf_accuracy * 100:.2f}%")
print(f"Training Time: {rf_time:.2f} seconds")

In [ ]:
# ============================================================
# CELL 19 - MODEL PERFORMANCE COMPARISON
# ============================================================

results = []

models_predictions = {
    "SVM": svm_pred,
    "KNN": knn_pred,
    "Decision Tree": dt_pred,
    "Random Forest": rf_pred
}

training_times = {
    "SVM": svm_time,
    "KNN": knn_time,
    "Decision Tree": dt_time,
    "Random Forest": rf_time
}

for model_name, predictions in models_predictions.items():

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    precision = precision_score(
        y_test,
        predictions,
        average='weighted',
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        average='weighted',
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        predictions,
        average='weighted',
        zero_division=0
    )

    results.append({
        "Model": model_name,
        "Accuracy (%)": accuracy * 100,
        "Precision (%)": precision * 100,
        "Recall (%)": recall * 100,
        "F1 Score (%)": f1 * 100,
        "Training Time (sec)": training_times[model_name]
    })

results_df = pd.DataFrame(results)

print("\nModel Performance:")
display(results_df)

In [ ]:
# ============================================================
# CELL 20 - RANDOM FOREST CLASSIFICATION REPORT
# ============================================================

print("Random Forest Classification Report:\n")

print(
    classification_report(
        y_test,
        rf_pred,
        target_names=selected_classes,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# CELL 21 - MAJORITY VOTING ENSEMBLE
# ============================================================

all_predictions = np.vstack([
    svm_pred,
    knn_pred,
    dt_pred,
    rf_pred
])

ensemble_pred = []

for i in range(all_predictions.shape[1]):

    votes = all_predictions[:, i]

    counts = np.bincount(votes)

    final_prediction = np.argmax(counts)

    ensemble_pred.append(final_prediction)

ensemble_pred = np.array(
    ensemble_pred
)

ensemble_accuracy = accuracy_score(
    y_test,
    ensemble_pred
)

print("Majority Voting Ensemble")
print("------------------------")
print(
    f"Ensemble Accuracy: "
    f"{ensemble_accuracy * 100:.2f}%"
)

correct = np.sum(
    ensemble_pred == y_test
)

incorrect = np.sum(
    ensemble_pred != y_test
)

print(f"Correct predictions: {correct}")
print(f"Incorrect predictions: {incorrect}")

In [ ]:
# ============================================================
# CELL 22 - ENSEMBLE CLASSIFICATION REPORT
# ============================================================

print("Ensemble Classification Report:\n")

print(
    classification_report(
        y_test,
        ensemble_pred,
        target_names=selected_classes,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# CELL 23 - ENSEMBLE CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    ensemble_pred
)

plt.figure(figsize=(10, 8))

plt.imshow(cm)

plt.title(
    "Confusion Matrix - Augmented Ensemble"
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")

plt.xticks(
    range(len(selected_classes)),
    selected_classes,
    rotation=45,
    ha='right'
)

plt.yticks(
    range(len(selected_classes)),
    selected_classes
)

plt.colorbar()

for i in range(cm.shape[0]):

    for j in range(cm.shape[1]):

        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 24 - INCORRECT ENSEMBLE PREDICTIONS
# ============================================================

incorrect_indices = np.where(
    ensemble_pred != y_test
)[0]

print(
    "Number of incorrect predictions:",
    len(incorrect_indices)
)

print(
    "Number of correct predictions:",
    len(y_test) - len(incorrect_indices)
)

print("\nSome incorrect predictions:")

for idx in incorrect_indices[:20]:

    print(
        f"Index: {idx} | "
        f"Actual: {selected_classes[y_test[idx]]} | "
        f"Predicted: {selected_classes[ensemble_pred[idx]]}"
    )

In [ ]:
# ============================================================
# CELL 25 - ORIGINAL VS AUGMENTED COMPARISON
# ============================================================

original_results = {
    "SVM": 98.99,
    "KNN": 95.50,
    "Decision Tree": 97.63,
    "Random Forest": 100.00,
    "Ensemble": 99.70
}

augmented_results = {
    "SVM": svm_accuracy * 100,
    "KNN": knn_accuracy * 100,
    "Decision Tree": dt_accuracy * 100,
    "Random Forest": rf_accuracy * 100,
    "Ensemble": ensemble_accuracy * 100
}

comparison_df = pd.DataFrame({
    "Model": list(original_results.keys()),
    "Original Accuracy (%)": list(original_results.values()),
    "Augmented Accuracy (%)": list(augmented_results.values())
})

comparison_df["Change (%)"] = (
    comparison_df["Augmented Accuracy (%)"]
    - comparison_df["Original Accuracy (%)"]
)

print("Original vs Augmented Model Performance")

display(comparison_df)

In [ ]:
# ============================================================
# CELL 26 - SAVE AUGMENTED MODELS
# ============================================================

os.makedirs(
    "models_augmented",
    exist_ok=True
)

joblib.dump(
    svm_model,
    "models_augmented/svm_model_augmented.pkl"
)

joblib.dump(
    knn_model,
    "models_augmented/knn_model_augmented.pkl"
)

joblib.dump(
    decision_tree_model,
    "models_augmented/decision_tree_model_augmented.pkl"
)

joblib.dump(
    random_forest_model,
    "models_augmented/random_forest_model_augmented.pkl"
)

joblib.dump(
    scaler,
    "models_augmented/scaler_augmented.pkl"
)

joblib.dump(
    selected_classes,
    "models_augmented/class_labels_augmented.pkl"
)

print("Augmented models saved successfully!")

print("\nSaved files:")

for filename in os.listdir("models_augmented"):
    print(" -", filename)

In [ ]:
# ============================================================
# CELL 27 - FINAL SUMMARY
# ============================================================

print("=" * 60)
print("AUGMENTED FRUITS-360 TRAINING SUMMARY")
print("=" * 60)

print(f"\nOriginal training images : {len(X_train)}")
print(f"Augmented images         : {len(X_augmented)}")
print(f"Final training images    : {len(X_train_final)}")
print(f"Test images              : {len(X_test)}")

print("\nModel Accuracies:")
print(f"SVM              : {svm_accuracy * 100:.2f}%")
print(f"KNN              : {knn_accuracy * 100:.2f}%")
print(f"Decision Tree    : {dt_accuracy * 100:.2f}%")
print(f"Random Forest    : {rf_accuracy * 100:.2f}%")
print(f"Ensemble         : {ensemble_accuracy * 100:.2f}%")

print("\nOriginal Baseline:")
print("SVM              : 98.99%")
print("KNN              : 95.50%")
print("Decision Tree    : 97.63%")
print("Random Forest    : 100.00%")
print("Ensemble         : 99.70%")

print("\n" + "=" * 60)
print("TRAINING COMPLETED")
print("=" * 60)